# 🛡️ Fine-Tuning YOLO26-nano on Real Human Faces + Web Privacy Elements

This notebook trains **Ultralytics YOLO26-nano (`yolo26n.pt`)** to detect:
- **`0: face`** $\to$ **Real photographic human faces** (WIDER FACE via `kagglehub`)
- **`1: password_field`** $\to$ **Password inputs, masked dot fields, credential forms**
- **`2: text_block`** $\to$ **Headings, form labels, privacy terms, button text**

The trained model is exported to **ONNX (Opset 12)** for direct in-browser inference in the **PS171 Privacy Agent** extension.

## ⚙️ Step 1: Install Dependencies & GPU Verification
Installs `ultralytics`, Google/Kaggle's modern `kagglehub` library, and ONNX tools.

In [ ]:
# [Cell 1] Install dependencies & auto-restart kernel
# In Colab's Python 3.13, restarting the session after installing Pillow avoids module cache errors.
!pip uninstall -y pillow
!pip install -q --no-cache-dir "pillow>=11.0.0" ultralytics onnx onnxsim opencv-python matplotlib pyyaml kagglehub

import os
print("\n" + "="*60)
print("✅ Packages installed successfully!")
print("🔄 Automatically restarting session to load fresh modules...")
print("👉 Wait 2 seconds until Colab reconnects, then proceed to Cell 2.")
print("="*60)
os.kill(os.getpid(), 9)

## 🩺 Step 1B: Verify Imports in Fresh Session
Run this cell after Cell 1 finishes restarting.

In [ ]:
# [Cell 2] Verify imports in the fresh kernel
import torch
import PIL
import kagglehub
import ultralytics
from ultralytics import YOLO

print(f"✅ PIL version: {PIL.__version__}")
print(f"✅ Ultralytics version: {ultralytics.__version__}")
print(f"✅ CUDA GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Model: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Running on CPU! Switch runtime to GPU under: Runtime > Change runtime type > T4 GPU.")

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully.")
except Exception:
    print("Drive mount skipped or running in local environment.")

## 📥 Step 2: Download Real Photographic Human Faces (WIDER FACE via KaggleHub)
Uses Kaggle's official `kagglehub` to download real human face photographs with ground-truth YOLO labels (`class 0: face`).

In [ ]:
# [Cell 2] Download Real Face Dataset
import kagglehub
import glob
import os
import shutil
from pathlib import Path

print("📥 Downloading WIDER FACE dataset via KaggleHub (high-speed Google Cloud backbone)...")
face_dataset_path = kagglehub.dataset_download("arnav109/wider-face-for-yolo-v8")
print(f"✅ Downloaded real face dataset to: {face_dataset_path}")

# Check available images in WIDER FACE
real_face_train_imgs = glob.glob(f"{face_dataset_path}/**/train/images/*.*", recursive=True)
real_face_val_imgs = glob.glob(f"{face_dataset_path}/**/val/images/*.*", recursive=True)
print(f"Found {len(real_face_train_imgs)} train and {len(real_face_val_imgs)} val real human face images.")

## 🌐 Step 3: Build Unified Dataset (Real Human Faces + Login Screens + Text Blocks)
Merges:
1. **Real Human Faces** $\to$ `class 0: face`
2. **Real & Synthetic Password Fields** $\to$ `class 1: password_field`
3. **UI Text Blocks** $\to$ `class 2: text_block`

In [ ]:
# [Cell 3] Unified Dataset Assembly with Real Human Faces
import os
import shutil
import random
from PIL import Image, ImageDraw

TARGET_DATASET = "/content/dataset"
shutil.rmtree(TARGET_DATASET, ignore_errors=True)

for split in ["train", "val"]:
    os.makedirs(f"{TARGET_DATASET}/images/{split}", exist_ok=True)
    os.makedirs(f"{TARGET_DATASET}/labels/{split}", exist_ok=True)

# --------------------------------------------------------------------------
# 1. Ingest Real Human Faces from WIDER FACE (Class 0)
# --------------------------------------------------------------------------
NUM_REAL_FACE_TRAIN = 400
NUM_REAL_FACE_VAL = 80

def ingest_real_faces(img_list, split, max_count):
    count = 0
    for img_p in img_list:
        if count >= max_count:
            break
        # Corresponding label file (.txt)
        lbl_p = img_p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        if not os.path.exists(lbl_p):
            continue
            
        dst_img = f"{TARGET_DATASET}/images/{split}/real_face_{count:04d}.jpg"
        dst_lbl = f"{TARGET_DATASET}/labels/{split}/real_face_{count:04d}.txt"
        
        shutil.copy2(img_p, dst_img)
        
        # Ensure all face labels are explicitly mapped to class 0
        with open(lbl_p, "r") as f_in:
            lines = f_in.readlines()
        
        remapped = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5:
                # Force class 0 (face)
                remapped.append("0 " + " ".join(parts[1:]))
                
        with open(dst_lbl, "w") as f_out:
            f_out.write("\n".join(remapped) + "\n")
            
        count += 1
    return count

ingested_train_faces = ingest_real_faces(real_face_train_imgs, "train", NUM_REAL_FACE_TRAIN)
ingested_val_faces = ingest_real_faces(real_face_val_imgs, "val", NUM_REAL_FACE_VAL)
print(f"✅ Ingested {ingested_train_faces} train and {ingested_val_faces} val real human face images.")

# --------------------------------------------------------------------------
# 2. Ingest Webpage Screenshots (Classes 1: password_field & 2: text_block)
# --------------------------------------------------------------------------
# Check if user uploaded synthetic_privacy_dataset.zip
zip_path = None
for candidate in ["/content/synthetic_privacy_dataset.zip", "/content/drive/MyDrive/synthetic_privacy_dataset.zip"]:
    if os.path.exists(candidate):
        zip_path = candidate
        break

if zip_path:
    import zipfile
    print(f"Extracting {zip_path}...")
    temp_dir = "/content/temp_synth"
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(temp_dir)
    for split in ["train", "val"]:
        s_imgs = glob.glob(f"{temp_dir}/images/{split}/*.*")
        for s_img in s_imgs:
            s_lbl = s_img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
            if os.path.exists(s_lbl):
                fname = os.path.basename(s_img)
                shutil.copy2(s_img, f"{TARGET_DATASET}/images/{split}/web_{fname}")
                shutil.copy2(s_lbl, f"{TARGET_DATASET}/labels/{split}/web_{os.path.splitext(fname)[0]}.txt")
    shutil.rmtree(temp_dir, ignore_errors=True)
    print("✅ Merged uploaded synthetic dataset.")
else:
    # Auto-generate webpage login screens if zip was not uploaded
    print("Generating webpage login screens with password fields & text blocks...")
    for split, count in [("train", 300), ("val", 60)]:
        for i in range(count):
            img = Image.new("RGB", (640, 640), (15, 23, 42) if random.random() > 0.5 else (245, 247, 250))
            draw = ImageDraw.Draw(img)
            boxes = []
            # Navbar
            draw.rectangle([0, 0, 640, 50], fill=(30, 41, 59))
            boxes.append((2, 0.15, 0.04, 0.2, 0.05)) # Nav title
            # Main Card
            draw.rectangle([140, 90, 500, 550], fill=(30, 41, 59), outline=(51, 65, 85))
            boxes.append((2, 0.5, 0.22, 0.45, 0.05)) # Card title
            boxes.append((2, 0.5, 0.28, 0.45, 0.04)) # Card sub
            # Username
            boxes.append((2, 0.3, 0.36, 0.2, 0.03))
            draw.rectangle([160, 245, 480, 285], fill=(15, 23, 42))
            # Password field (1)
            boxes.append((2, 0.25, 0.47, 0.15, 0.03))
            draw.rectangle([160, 315, 480, 355], fill=(15, 23, 42), outline=(59, 130, 246))
            boxes.append((1, 0.5, 0.523, 0.5, 0.0625))
            # Submit button (2)
            draw.rectangle([160, 410, 480, 455], fill=(59, 130, 246))
            boxes.append((2, 0.5, 0.675, 0.5, 0.07))
            
            dst_img = f"{TARGET_DATASET}/images/{split}/gen_screen_{i:04d}.jpg"
            dst_lbl = f"{TARGET_DATASET}/labels/{split}/gen_screen_{i:04d}.txt"
            img.save(dst_img, quality=95)
            with open(dst_lbl, "w") as f:
                for cls_id, xc, yc, bw, bh in boxes:
                    f.write(f"{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

print(f"\n🎉 Final Unified Dataset: "
      f"{len(os.listdir(TARGET_DATASET + '/images/train'))} train images, "
      f"{len(os.listdir(TARGET_DATASET + '/images/val'))} val images.")

## 📝 Step 4: Write `data.yaml` Configuration

In [ ]:
# [Cell 4] Write data.yaml
import yaml

yaml_data = {
    'path': '/content/dataset',
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'face',
        1: 'password_field',
        2: 'text_block'
    }
}

yaml_path = '/content/dataset/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_data, f, sort_keys=False)

print(f"Created config at {yaml_path}:")
!cat /content/dataset/data.yaml

## 🚀 Step 5: Fine-Tune YOLO26-nano (`yolo26n.pt`)

### Hyperparameter Recipe for Faces + UI Elements:
- **`fliplr=0.5`**: Enables horizontal flipping for human face diversity.
- **`degrees=0.0` & `flipud=0.0`**: Prevents upside-down or diagonal webpage distortions.
- **`mosaic=0.5`**: Blends faces and webpage components.
- **`lr0=0.005` & `patience=15`**: Fast convergence without overfitting.

In [ ]:
# [Cell 5] Fine-Tune YOLO26-nano
from ultralytics import YOLO

# 1. Initialize YOLO26-nano base checkpoint
model = YOLO('yolo26n.pt')

# 2. Train on unified dataset
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=50,
    patience=15,
    batch=16,
    imgsz=640,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=4,
    optimizer='AdamW',
    lr0=0.005,
    lrf=0.01,
    warmup_epochs=3.0,
    # Augmentations for both human faces & UI elements
    degrees=0.0,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.5,
    mixup=0.0,
    project='/content/runs/train',
    name='yolo26n_privacy_faces',
    save=True,
    val=True
)

# Save best checkpoint to Google Drive
best_pt_path = '/content/runs/train/yolo26n_privacy_faces/weights/best.pt'
if os.path.exists('/content/drive/MyDrive') and os.path.exists(best_pt_path):
    !cp {best_pt_path} /content/drive/MyDrive/yolo26n_privacy_best.pt
    print("✅ Saved best.pt to Google Drive: /content/drive/MyDrive/yolo26n_privacy_best.pt")

## 📦 Step 6: Export to ONNX (Opset 12 Web Standard)
Exports model in **Opset 12** for direct in-browser inference in the Chrome/Brave extension.

In [ ]:
# [Cell 6] Export to Opset 12 ONNX
from ultralytics import YOLO
import os

best_pt_path = '/content/runs/train/yolo26n_privacy_faces/weights/best.pt'
trained_model = YOLO(best_pt_path)

# Export with Opset 12 (Universal WebAssembly browser standard)
onnx_export_path = trained_model.export(
    format='onnx',
    imgsz=640,
    opset=12,
    simplify=True,
    dynamic=False
)

print(f"\n🎉 Successfully exported ONNX model: {onnx_export_path}")

# Copy to Google Drive
if os.path.exists('/content/drive/MyDrive'):
    !cp {onnx_export_path} /content/drive/MyDrive/yolo26n_privacy_web.onnx
    print("✅ Copied ONNX to Google Drive: /content/drive/MyDrive/yolo26n_privacy_web.onnx")

# Provide direct browser download button
from google.colab import files
files.download(onnx_export_path)

## 🔍 Step 7: Visual Sanity Check on Real Human Faces & Login Screens
Plots color-coded bounding boxes on validation images to confirm detection quality.

In [ ]:
# [Cell 7] Visual Validation
import glob
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

val_model = YOLO('/content/runs/train/yolo26n_privacy_faces/weights/best.pt')

# Pick sample images from both real faces and web screens
val_images = glob.glob('/content/dataset/images/val/*.*')[:6]

if val_images:
    results = val_model.predict(source=val_images, conf=0.25, imgsz=640, device=0 if torch.cuda.is_available() else 'cpu')
    
    num_imgs = len(val_images)
    cols = min(3, num_imgs)
    rows = (num_imgs + cols - 1) // cols
    fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(6 * cols, 5 * rows))
    if num_imgs == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    for idx, (img_path, r) in enumerate(zip(val_images, results)):
        annotated_bgr = r.plot()
        annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(annotated_rgb)
        axes[idx].set_title(os.path.basename(img_path), fontsize=10)
        axes[idx].axis('off')
        
    for j in range(idx + 1, len(axes)):
        axes[j].axis('off')
        
    plt.tight_layout()
    plt.show()